# Task A Evaluation — User Modeling
### Team Greenlight · DSN x BCT LLM Agent Hackathon 3.0

Metrics computed:
- **ROUGE-L** — overlap between generated and real review text
- **RMSE** — error between predicted and actual star rating

Results saved to `results/task_a_results.json`

In [1]:
import sys, json, random
from pathlib import Path

ROOT = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(ROOT / "task_a"))

import numpy as np
from rouge_score import rouge_scorer
from agent import generate_review, _load_profiles

DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Imports OK")

C:\Users\Ayoola\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\Ayoola\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports OK


In [2]:
# Load data
with open(DATA_DIR / "combined_sample.json", encoding="utf-8") as f:
    all_records = json.load(f)

profiles = _load_profiles()
print(f"Records: {len(all_records):,}")
print(f"Profiles: {len(profiles):,}")

# Index records by user_id
from collections import defaultdict
user_records = defaultdict(list)
for r in all_records:
    user_records[r["user_id"]].append(r)

# Pick 20 users who have at least 2 reviews (need one as ground truth)
eligible = [
    uid for uid, recs in user_records.items()
    if uid in profiles and len(recs) >= 2
]
random.seed(42)
test_users = random.sample(eligible, min(20, len(eligible)))
print(f"\nTest users selected: {len(test_users)}")

INFO:agent:Loaded 1,271 user profiles.


Records: 9,999
Profiles: 1,271

Test users selected: 20


In [3]:
# Run evaluation
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
results = []

for i, uid in enumerate(test_users):
    # Use the last review as ground truth
    ground_truth = user_records[uid][-1]

    print(f"[{i+1}/{len(test_users)}] User: {uid}")
    try:
        output = generate_review(
            user_id=uid,
            item_name=ground_truth["item_name"],
            item_category=ground_truth["item_category"],
            item_description=ground_truth["review_text"][:100],
            price_range="Unknown",
        )

        # ROUGE-L
        rouge = scorer.score(
            ground_truth["review_text"],
            output["review_text"]
        )
        rouge_l = round(rouge["rougeL"].fmeasure, 4)

        # Rating error
        actual_rating    = ground_truth["rating"]
        predicted_rating = output["star_rating"]
        rating_error     = abs(predicted_rating - actual_rating)

        result = {
            "user_id":           uid,
            "item_name":         ground_truth["item_name"],
            "item_category":     ground_truth["item_category"],
            "actual_rating":     actual_rating,
            "predicted_rating":  predicted_rating,
            "rating_error":      round(rating_error, 4),
            "rouge_l":           rouge_l,
            "review_length":     len(output["review_text"]),
            "user_style_matched": output["user_style_matched"],
            "generated_review":  output["review_text"][:300],
            "actual_review":     ground_truth["review_text"][:300],
        }
        results.append(result)
        print(f"  ROUGE-L: {rouge_l:.4f}  |  Rating error: {rating_error:.2f}  "
              f"(predicted={predicted_rating}, actual={actual_rating})")

    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({"user_id": uid, "error": str(e)})

print(f"\nCompleted {len(results)} evaluations.")

INFO:absl:Using default tokenizer.


[1/20] User: yelp_user_0228


INFO:agent:Finish reason: STOP
INFO:agent:Generated review for yelp_user_0228 | rating=3.9 | confidence=0.78


  ROUGE-L: 0.1193  |  Rating error: 0.90  (predicted=3.9, actual=3.0)
[2/20] User: yelp_user_0051


INFO:agent:Finish reason: STOP
INFO:agent:Generated review for yelp_user_0051 | rating=4.2 | confidence=0.8


...

  ROUGE-L: 0.1871  |  Rating error: 0.00  (predicted=5.0, actual=5.0)
[19/20] User: yelp_user_0054


INFO:agent:Finish reason: MAX_TOKENS
INFO:agent:Generated review for yelp_user_0054 | rating=3.7 | confidence=0.77


  ROUGE-L: 0.0905  |  Rating error: 0.30  (predicted=3.7, actual=4.0)
[20/20] User: amazon_AH2IW7MZIXM53IRDZ2XY


INFO:agent:Finish reason: STOP
INFO:agent:Generated review for amazon_AH2IW7MZIXM53IRDZ2XY | rating=5.0 | confidence=0.83


  ROUGE-L: 0.2154  |  Rating error: 0.00  (predicted=5.0, actual=5.0)

Completed 20 evaluations.


In [4]:
# Compute aggregate metrics
valid = [r for r in results if "rouge_l" in r]

rouge_scores  = [r["rouge_l"]       for r in valid]
rating_errors = [r["rating_error"]  for r in valid]

avg_rouge = np.mean(rouge_scores)
rmse      = np.sqrt(np.mean(np.array(rating_errors) ** 2))
mae       = np.mean(rating_errors)
style_matched = sum(1 for r in valid if r.get("user_style_matched")) / len(valid)

summary = {
    "num_users_evaluated": len(valid),
    "avg_rouge_l":         round(float(avg_rouge), 4),
    "rmse":                round(float(rmse), 4),
    "mae":                 round(float(mae), 4),
    "style_match_rate":    round(style_matched, 4),
    "avg_review_length":   round(float(np.mean([r["review_length"] for r in valid])), 1),
    "individual_results":  results,
}

with open(RESULTS_DIR / "task_a_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("=" * 50)
print("TASK A EVALUATION RESULTS")
print("=" * 50)
print(f"  Users evaluated  : {len(valid)}")
print(f"  Avg ROUGE-L      : {avg_rouge:.4f}")
print(f"  RMSE (rating)    : {rmse:.4f}")
print(f"  MAE  (rating)    : {mae:.4f}")
print(f"  Style match rate : {style_matched:.1%}")
print(f"  Avg review length: {summary['avg_review_length']:.0f} chars")
print(f"\nSaved → results/task_a_results.json")

TASK A EVALUATION RESULTS
  Users evaluated  : 16
  Avg ROUGE-L      : 0.1357
  RMSE (rating)    : 0.8725
  MAE  (rating)    : 0.6625
  Style match rate : 100.0%
  Avg review length: 1142 chars

Saved → results/task_a_results.json


In [5]:
# Print summary table
print(f"{'User ID':<25} {'ROUGE-L':>8} {'Pred':>6} {'Actual':>7} {'Error':>6}")
print("-" * 57)
for r in valid:
    print(f"{r['user_id']:<25} {r['rouge_l']:>8.4f} "
          f"{r['predicted_rating']:>6.1f} {r['actual_rating']:>7.1f} "
          f"{r['rating_error']:>6.2f}")
print("-" * 57)
print(f"{'AVERAGE':<25} {avg_rouge:>8.4f} {'':>6} {'':>7} {mae:>6.2f}")
print(f"{'RMSE':<25} {'':>8} {'':>6} {'':>7} {rmse:>6.4f}")

User ID                    ROUGE-L   Pred  Actual  Error
---------------------------------------------------------
yelp_user_0228              0.1193    3.9     3.0   0.90
yelp_user_0051              0.1206    4.2     4.0   0.20
yelp_user_0563              0.1453    3.8     4.0   0.20
yelp_user_0501              0.1176    4.1     2.0   2.10
yelp_user_0457              0.1086    4.0     3.0   1.00
yelp_user_0285              0.1296    3.0     2.0   1.00
yelp_user_0209              0.1270    5.0     4.0   1.00
amazon_AGBJABSJLN7H7AALE7VK   0.2936    5.0     5.0   0.00
yelp_user_0178              0.0818    3.4     3.0   0.40
yelp_user_0191              0.0754    4.3     3.0   1.30
yelp_user_0447              0.1160    4.1     3.0   1.10
yelp_user_0476              0.1289    4.8     4.0   0.80
amazon_AEVPPTMG43C6GWSR7I2U   0.1143    4.7     5.0   0.30
gr_AH4O5W3EM4CKQGHMBVTS     0.1871    5.0     5.0   0.00
yelp_user_0054              0.0905    3.7     4.0   0.30
amazon_AH2IW7MZIXM53IRDZ2X